# Text Encoding Pipeline

This notebook prepares the review dataset, cleans and lemmatizes the text, and then applies three common text encoding techniques:

- One-Hot Encoding
- Bag of Words
- TF-IDF


In [33]:
import re
import string
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split


## 1. Load the Dataset

We load the raw review file and inspect the first few rows.


In [34]:
data_path = Path.cwd() / "data" / "Reviews.csv"
if not data_path.exists():
    data_path = Path.cwd().parent / "data" / "Reviews.csv"

df = pd.read_csv(data_path)

In [35]:
df.sample(10)

,Star Rating,Review Title,Review Description
616,1.0 out of 5 stars,Please don't buy,This is totally damaged product and packing is...
1158,4.0 out of 5 stars,Battery,This price range phone is good. Camera good. D...
787,3.0 out of 5 stars,Good as per the price. Good sound,Slow Processing. Sound is good.Read more
1413,5.0 out of 5 stars,NaN,ExlantRead more
785,5.0 out of 5 stars,Good picture quality,Very good product at this price range.Must buy...
766,5.0 out of 5 stars,Great Deal,The media could not be loaded.I bought TCL T8C...
370,5.0 out of 5 stars,Amazing product.,"I like the airdopes very much, The sound quali..."
627,3.0 out of 5 stars,Lacks punch,It lacks punching bassRead more
1084,4.0 out of 5 stars,Satisfied with the product,This product is very nice and value for moneRe...
535,5.0 out of 5 stars,Satisfied good,Good Goooood GoooooooodThis Clip earbuds hav...


In [36]:
df.shape


(1500, 3)

## 2. Create the Working DataFrame

We combine the title and description into a single review column, create sentiment labels from star ratings, remove duplicates, and keep only the columns needed for modeling.


In [37]:
def map_sentiment(rating_text):
    rating = int(re.search(r"\d+", str(rating_text)).group())
    if rating >= 4:
        return "positive"
    if rating == 3:
        return "neutral"
    return "negative"


df["Review Title"] = df["Review Title"].fillna("")
df["Review Description"] = df["Review Description"].fillna("")
df["review_text"] = (df["Review Title"] + " " + df["Review Description"]).str.strip()
df["sentiment_label"] = df["Star Rating"].apply(map_sentiment)

df = df[["review_text", "sentiment_label"]].copy()
df = df.drop_duplicates(subset=["review_text"]).reset_index(drop=True)

df.head()


,review_text,sentiment_label
0,"Great Deal, Smooth Delivery & Premium Experien...",positive
1,"Solid Phone, Solid Performance and premium exp...",positive
2,Premium and class as always. Very Premium feel...,positive
3,Performance First time i have shifted from And...,positive
4,Delivery and item performance is superfine Per...,positive


In [38]:
df["sentiment_label"].value_counts()


sentiment_label
positive    718
negative    102
neutral      81
Name: count, dtype: int64

## 3. Clean and Lemmatize the Text

This section lowercases the text, removes punctuation, removes stopwords, lemmatizes each token, and stores the cleaned result back in the dataframe.


In [39]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\piyus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\piyus\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\piyus\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\piyus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [40]:
def clean_and_lemmatize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words and token.isalnum()]
    lemmas = [lemmatizer.lemmatize(token, pos="v") for token in tokens]

    return " ".join(lemmas)


df["cleaned_review_text"] = df["review_text"].apply(clean_and_lemmatize)
df.head()


,review_text,sentiment_label,cleaned_review_text
0,"Great Deal, Smooth Delivery & Premium Experien...",positive,great deal smooth delivery premium experience ...
1,"Solid Phone, Solid Performance and premium exp...",positive,solid phone solid performance premium experien...
2,Premium and class as always. Very Premium feel...,positive,premium class always premium feel understand a...
3,Performance First time i have shifted from And...,positive,performance first time shift android ios iphon...
4,Delivery and item performance is superfine Per...,positive,delivery item performance superfine perfect sp...


In [41]:
df = df[df["cleaned_review_text"].str.len() > 0].reset_index(drop=True)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment_label"])

df.head()


,review_text,sentiment_label,cleaned_review_text,sentiment_encoded
0,"Great Deal, Smooth Delivery & Premium Experien...",positive,great deal smooth delivery premium experience ...,2
1,"Solid Phone, Solid Performance and premium exp...",positive,solid phone solid performance premium experien...,2
2,Premium and class as always. Very Premium feel...,positive,premium class always premium feel understand a...,2
3,Performance First time i have shifted from And...,positive,performance first time shift android ios iphon...,2
4,Delivery and item performance is superfine Per...,positive,delivery item performance superfine perfect sp...,2


## 4. Word Frequency After Lemmatization

We count the most common normalized words to inspect the cleaned corpus.


In [42]:
all_lemmas = " ".join(df["cleaned_review_text"]).split()
lemma_counts = Counter(all_lemmas)
lemma_counts.most_common(20)


[('good', 992),
 ('quality', 721),
 ('sound', 477),
 ('phone', 466),
 ('use', 437),
 ('product', 397),
 ('battery', 360),
 ('tv', 348),
 ('price', 318),
 ('great', 246),
 ('earbuds', 241),
 ('also', 228),
 ('work', 227),
 ('feel', 211),
 ('value', 211),
 ('buy', 208),
 ('charge', 195),
 ('look', 189),
 ('performance', 187),
 ('best', 183)]

## 5. One-Hot Encoding

A binary `CountVectorizer` creates a one-hot style representation where each selected word is marked as present or absent in a review.


In [43]:
ohe_vectorizer = CountVectorizer(binary=True, max_features=30)
ohe_matrix = ohe_vectorizer.fit_transform(df["cleaned_review_text"])

ohe_df = pd.DataFrame(
    ohe_matrix.toarray(),
    columns=ohe_vectorizer.get_feature_names_out()
)

ohe_df.head()


,also,battery,best,buy,call,camera,charge,clear,earbuds,excellent,...,performance,phone,price,product,quality,sound,tv,use,value,work
0,1,0,0,1,0,1,0,0,0,0,...,1,0,0,1,1,0,0,1,0,0
1,1,0,0,1,0,1,0,0,0,1,...,1,1,0,1,1,0,0,1,0,0
2,1,1,0,1,0,1,0,0,0,1,...,0,1,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


## 6. Bag of Words

Bag of Words counts how many times each selected word appears in every review.


In [44]:
bow_vectorizer = CountVectorizer(max_features=30)
bow_matrix = bow_vectorizer.fit_transform(df["cleaned_review_text"])

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

bow_df.head()


,also,battery,best,bud,buy,call,camera,charge,clear,earbuds,...,performance,phone,price,product,quality,sound,tv,use,value,work
0,1,0,0,0,1,0,1,0,0,0,...,1,0,0,1,1,0,0,1,0,0
1,1,0,0,0,1,0,1,0,0,0,...,2,1,0,1,1,0,0,1,0,0
2,1,2,0,0,1,0,1,0,0,0,...,0,1,0,0,0,0,0,2,0,0
3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


## 7. TF-IDF

TF-IDF highlights words that are important in a review while reducing the weight of very common words.


In [45]:
tfidf_vectorizer = TfidfVectorizer(max_features=30)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["cleaned_review_text"])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.head()


,also,battery,best,bud,buy,call,camera,charge,clear,earbuds,...,performance,phone,price,product,quality,sound,tv,use,value,work
0,0.210999,0.000000,0.0,0.0,0.222459,0.0,0.240431,0.0,0.0,0.0,...,0.228586,0.000000,0.0,0.172446,0.138399,0.0,0.0,0.178369,0.0,0.0
1,0.196654,0.000000,0.0,0.0,0.207335,0.0,0.224085,0.0,0.0,0.0,...,0.426092,0.182171,0.0,0.160722,0.128990,0.0,0.0,0.166243,0.0,0.0
2,0.249359,0.423053,0.0,0.0,0.262902,0.0,0.284141,0.0,0.0,0.0,...,0.000000,0.230994,0.0,0.000000,0.000000,0.0,0.0,0.421593,0.0,0.0
3,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.382541,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0
4,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,1.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0


## 8. Final DataFrames

The cleaned dataframe and the three encoded dataframes are now ready for further modeling.


In [46]:
processed_df = df[["review_text", "cleaned_review_text", "sentiment_label", "sentiment_encoded"]].copy()

processed_df.head()


,review_text,cleaned_review_text,sentiment_label,sentiment_encoded
0,"Great Deal, Smooth Delivery & Premium Experien...",great deal smooth delivery premium experience ...,positive,2
1,"Solid Phone, Solid Performance and premium exp...",solid phone solid performance premium experien...,positive,2
2,Premium and class as always. Very Premium feel...,premium class always premium feel understand a...,positive,2
3,Performance First time i have shifted from And...,performance first time shift android ios iphon...,positive,2
4,Delivery and item performance is superfine Per...,delivery item performance superfine perfect sp...,positive,2


In [47]:
processed_df.shape, ohe_df.shape, bow_df.shape, tfidf_df.shape


((898, 4), (898, 30), (898, 30), (898, 30))

## 9. Expand the Sentiment Lexicon from Review Vocabulary

This section keeps the original lexicon idea, but improves it by using `Counter` and `CountVectorizer` on `Review Title` and `Review Description` to discover frequent sentiment-related words and phrases from the dataset itself.


In [48]:
# Rewrite the manual sentiment-word extraction using vocabulary from the corpus
lexicon_df = pd.read_csv(data_path)
lexicon_df["Review Title"] = lexicon_df["Review Title"].fillna("")
lexicon_df["Review Description"] = lexicon_df["Review Description"].fillna("")

lexicon_df["title_description"] = (
    lexicon_df["Review Title"] + " " + lexicon_df["Review Description"]
).str.strip()

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

lexicon_df["normalized_text"] = lexicon_df["title_description"].apply(normalize_text)

all_words = " ".join(lexicon_df["normalized_text"]).split()
word_counter = Counter(
    word for word in all_words
    if len(word) > 2 and word not in stop_words
)

# Most frequent vocabulary from the review corpus
top_vocab = [word for word, count in word_counter.most_common(150)]

# Use CountVectorizer to find common words and phrases from title + description
phrase_vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), max_features=400)
phrase_matrix = phrase_vectorizer.fit_transform(lexicon_df["normalized_text"])
phrase_counts = pd.DataFrame({
    "term": phrase_vectorizer.get_feature_names_out(),
    "count": phrase_matrix.sum(axis=0).A1
}).sort_values("count", ascending=False)

# Seed sentiment lexicon
seed_sentiment_phrases = [
    "good", "bad", "excellent", "worst", "poor", "amazing", "awesome",
    "not good", "not bad", "value for money", "disappointed", "best",
    "nice", "comfortable", "sturdy", "strong", "problem", "issue",
    "waste of money", "great", "love", "perfect", "easy", "happy",
    "satisfied", "recommend", "durable", "stable", "cheap", "broken",
    "hard", "difficult", "return", "defective","refund", "expensive", "flimsy"
]

# Broader sentiment clue words used to pull more useful terms from the corpus
sentiment_clues = {
    "good", "great", "best", "nice", "excellent", "amazing", "awesome",
    "perfect", "love", "happy", "satisfied", "recommend", "comfortable",
    "sturdy", "strong", "durable", "stable", "easy", "worth", "value",
    "bad", "worst", "poor", "disappointed", "problem", "issue", "broken",
    "hard", "difficult", "cheap", "expensive", "return", "refund", "waste",
    "flimsy", "defective", "uncomfortable"
}

auto_sentiment_words = [
    word for word in top_vocab
    if word in sentiment_clues
]

auto_sentiment_phrases = [
    term for term in phrase_counts["term"].tolist()
    if any(token in sentiment_clues for token in term.split())
][:60]

sentiment_phrases = sorted(set(
    seed_sentiment_phrases +
    auto_sentiment_words +
    auto_sentiment_phrases
))

def extract_sentiment_words(text):
    if pd.isna(text):
        return ""

    text = normalize_text(text)
    found = []

    for phrase in sentiment_phrases:
        if phrase in text:
            found.append(phrase)

    return ", ".join(found)

# Apply on both Review Title and Review Description together
lexicon_df["sentiment_words"] = lexicon_df["title_description"].apply(extract_sentiment_words)

print("Top frequent words from corpus:")
print(word_counter.most_common(20))
print("\nExpanded sentiment phrase count:", len(sentiment_phrases))
print("\nSample output:")
print(lexicon_df[["Review Title", "Review Description", "sentiment_words"]].head())


Top frequent words from corpus:
[('good', 1880), ('quality', 1306), ('read', 936), ('sound', 866), ('phone', 865), ('battery', 777), ('product', 775), ('price', 633), ('great', 523), ('use', 497), ('earbuds', 439), ('performance', 430), ('value', 418), ('overall', 414), ('camera', 411), ('also', 400), ('money', 357), ('best', 337), ('clear', 301), ('experience', 297)]

Expanded sentiment phrase count: 61

Sample output:
                                        Review Title  \
0   Great Deal, Smooth Delivery & Premium Experience   
1  Solid Phone, Solid Performance and premium exp...   
2                       Premium and class as always.   
3                                        Performance   
4         Delivery and item performance is superfine   

                                  Review Description  \
0  I recently purchased the iPhone 17 Pro from Am...   
1  I recently purchased the iPhone 17 Pro from Am...   
2  Very Premium feel. I do understand that apple ...   
3  First time i

In [49]:
lexicon_df["Review Description"][29]

'This phone is really excellent. The camera is outstanding — I compared it side by side with other phones around the ₹30,000 range, but this one clearly has a much better camera. The pictures are very clear with great quality, and the OIS works really well. It is visibly better than other brands.The display is also excellent. The colors are very vibrant and the screen is extremely smooth. Using the phone feels very enjoyable and smooth.The battery performance is also very good. One full charge easily lasts a full day even with heavy usage.Apps open very fast and the phone feels very smooth overall. Games like BGMI also run smoothly; I have played it without any issues.One small drawback is that it has a single speaker. However, it is not bad at all — the speaker is loud and the sound quality is good. Still, if it had stereo speakers, it would have been even better.I am giving this phone 4 stars only because of the single speaker.Overall phone is very good and Balanced around 25,000.Rea

## 10. Train a Classifier on `sentiment_words` and Predict User Comments

This section creates a classification dataset where `sentiment_words` is the feature and `sentiment_label` is the target, trains a model, and then predicts sentiment for a new user comment after extracting its sentiment words.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import MultinomialNB,BernoulliNB
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [51]:
lexicon_df["sentiment_label"] = lexicon_df["Star Rating"].apply(map_sentiment)

classification_df = lexicon_df[["title_description", "sentiment_words", "sentiment_label"]].copy()
classification_df["sentiment_words"] = classification_df["sentiment_words"].fillna("")
classification_df = classification_df[classification_df["sentiment_words"].str.strip() != ""].reset_index(drop=True)

X = classification_df["sentiment_words"]
y = classification_df["sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

sentiment_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_vec = sentiment_vectorizer.fit_transform(X_train)
X_test_vec = sentiment_vectorizer.transform(X_test)

sentiment_model = LogisticRegression(max_iter=1000)
sentiment_model.fit(X_train_vec, y_train)

y_pred = sentiment_model.predict(X_test_vec)

print("Model accuracy:", round(accuracy_score(y_test, y_pred), 4))



Model accuracy: 0.8944


In [ ]:
mnb = MultinomialNB()
bnb = BernoulliNB()

In [55]:
sentiment_model = MultinomialNB()
sentiment_model.fit(X_train_vec, y_train)

y_pred = sentiment_model.predict(X_test_vec)

print("Model accuracy:", round(accuracy_score(y_test, y_pred), 4))

Model accuracy: 0.8873


In [56]:
sentiment_model = BernoulliNB()
sentiment_model.fit(X_train_vec, y_train)

y_pred = sentiment_model.predict(X_test_vec)

print("Model accuracy:", round(accuracy_score(y_test, y_pred), 4))

Model accuracy: 0.8662


In [57]:
def predict_sentiment_from_comment(comment):
    extracted = extract_sentiment_words(comment)
    extracted_words = [word.strip() for word in extracted.split(",") if word.strip()]

    if extracted_words:
        feature_text = ", ".join(extracted_words)
        feature_vector = sentiment_vectorizer.transform([feature_text])
        predicted_label = sentiment_model.predict(feature_vector)[0]
    else:
        feature_text = ""
        predicted_label = "neutral"

    return {
        "comment": comment,
        "extracted_sentiment_words": extracted_words,
        "model_input": feature_text,
        "predicted_sentiment": predicted_label
    }


In [58]:
# Example user comment
user_comment = "I’ve been using the OnePlus 15R for some time now, and it’s a really solid all-rounder with strong performance and reliable battery life.👍 What I liked:Fast & smooth performance: Handles everything easily — multitasking, gaming, daily use, all super fluid.No heating issues: Even during longer gaming sessions, the phone stays stable without noticeable heating.Great display: The AMOLED screen is bright, smooth, and perfect for content and gaming.Good main camera: The primary lens performs really well and captures sharp, detailed photos. It’s actually the same main sensor as the OnePlus 15, which is impressive at this price.Strong battery life: Easily lasts a full day with heavy usage.Clean software: OxygenOS is smooth and user-friendly.👎 What could be better:No telephoto lens: This is the biggest downside. While the main camera is great, the lack of a dedicated telephoto camera is disappointing for zoom shots.No wireless charging: Would have made it more complete.🤔 Final verdict:The OnePlus 15R is a powerful and reliable smartphone with great performance, solid battery life, and a surprisingly good main camera. The only real drawback is the missing telephoto lens.👉 Best for: Performance users, gamers, and casual photography👉 Not ideal for: People who want advanced zoom photographyEven though I used Chatgpt to write this review but written information is given by me with personal experience 😁Read more"

prediction = predict_sentiment_from_comment(user_comment)
pd.DataFrame([prediction])

,comment,extracted_sentiment_words,model_input,predicted_sentiment
0,I’ve been using the OnePlus 15R for some time ...,"[best, good, great, issue, perfect, stable, st...","best, good, great, issue, perfect, stable, strong",positive
